In [2]:
# pull the cfb api key from the .env file
import os
from dotenv import load_dotenv, find_dotenv
import cfbd
import pandas as pd
import numpy as np

load_dotenv(find_dotenv())  # go up one directory

CFB_API_KEY = os.getenv('CFBD_API_KEY')

In [4]:
import cfbd
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

configuration = cfbd.Configuration()
configuration.access_token = os.environ["CFBD_API_KEY"]

api_client = cfbd.ApiClient(configuration)
players_api = cfbd.PlayersApi(api_client)
stats_api = cfbd.StatsApi(api_client)



## Data Fetching Exploration

In [5]:
# import cfbd
# import os
# from dotenv import load_dotenv, find_dotenv

# load_dotenv(find_dotenv())

# configuration = cfbd.Configuration()
# configuration.access_token = os.environ["CFBD_API_KEY"]

# api_client = cfbd.ApiClient(configuration)
# players_api = cfbd.PlayersApi(api_client)
# stats_api = cfbd.StatsApi(api_client)

# results = players_api.search_players(search_term="Bryce Young")
# usage = players_api.get_player_usage(year=2024, team="Alabama")
# stats = stats_api.get_player_season_stats(year=2024, team="Alabama")
# returning = players_api.get_returning_production(year=2024, team="Alabama")
# transfers = players_api.get_transfer_portal(year=2024)

# print(results)
# print(usage)
# print(stats)

In [6]:
# import matplotlib.pyplot as plt

# # Fetch 2024 passing and rushing stats across all teams
# passing_raw = stats_api.get_player_season_stats(year=2024, category='passing')
# rushing_raw = stats_api.get_player_season_stats(year=2024, category='rushing')

# def stats_to_wide(stats_list):
#     records = [{
#         'player': s.player, 'player_id': s.player_id,
#         'position': s.position, 'team': s.team,
#         'conference': s.conference, 'stat_type': s.stat_type,
#         'stat': float(s.stat)
#     } for s in stats_list]
#     df = pd.DataFrame(records)
#     return df.pivot_table(
#         index=['player', 'player_id', 'position', 'team', 'conference'],
#         columns='stat_type', values='stat'
#     ).reset_index()

# pass_df = stats_to_wide(passing_raw)
# rush_df = stats_to_wide(rushing_raw)

# # Filter to meaningful sample sizes
# qbs = pass_df[pass_df['ATT'] >= 100].copy()
# rbs = rush_df[(rush_df['position'] == 'RB') & (rush_df['CAR'] >= 50)].copy()

# # PCT is stored as decimal (0.0–1.0)
# qbs['COMP_PCT'] = qbs['PCT'] * 100

# fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# fig.suptitle('2024 College Football Player Stats', fontsize=15, fontweight='bold')

# # QB passing yards distribution
# axes[0, 0].hist(qbs['YDS'], bins=30, color='steelblue', edgecolor='black')
# axes[0, 0].set_title('QB Passing Yards (min. 100 attempts)')
# axes[0, 0].set_xlabel('Passing Yards')
# axes[0, 0].set_ylabel('# of QBs')

# # QB TD vs INT scatter
# axes[0, 1].scatter(qbs['INT'], qbs['TD'], alpha=0.6, color='darkorange', edgecolors='black', linewidths=0.4)
# axes[0, 1].set_title('QB Touchdowns vs Interceptions')
# axes[0, 1].set_xlabel('Interceptions')
# axes[0, 1].set_ylabel('Touchdowns')

# # QB completion % distribution
# axes[1, 0].hist(qbs['COMP_PCT'], bins=25, color='seagreen', edgecolor='black')
# axes[1, 0].set_title('QB Completion % (min. 100 attempts)')
# axes[1, 0].set_xlabel('Completion %')
# axes[1, 0].set_ylabel('# of QBs')

# # RB yards per carry distribution
# axes[1, 1].hist(rbs['YPC'], bins=25, color='mediumpurple', edgecolor='black')
# axes[1, 1].set_title('RB Yards Per Carry (min. 50 carries)')
# axes[1, 1].set_xlabel('Yards Per Carry')
# axes[1, 1].set_ylabel('# of RBs')

# plt.tight_layout()
# plt.show()

In [7]:
# import seaborn as sns

# fig, ax = plt.subplots(figsize=(8, 5))
# sns.histplot(qbs['COMP_PCT'], bins=25, kde=True, color='steelblue', ax=ax, edgecolor='white' )
# ax.set_title('QB Completion % — 2024 (min. 100 attempts)')
# ax.set_xlabel('Completion %')
# ax.set_ylabel('# of QBs')
# plt.tight_layout()
# plt.show()

In [ ]:
# Pull all 2023 player season stats from /stats/player/season
all_stats_raw = stats_api.get_player_season_stats(year=2023)

season_df = pd.DataFrame([{
    'player_id':  s.player_id,
    'player':     s.player,
    'position':   s.position,
    'team':       s.team,
    'conference': s.conference,
    'category':   s.category,
    'stat_type':  s.stat_type,
    'stat':       s.stat,
} for s in all_stats_raw])

print(f"Total rows: {len(season_df)}")
print(f"Categories: {sorted(season_df['category'].unique())}")
print(f"Teams: {season_df['team'].nunique()}")
season_df.head(20)

In [16]:
import glob

av_files = glob.glob('../scraping_av/data/*.csv')
av_df = pd.concat([pd.read_csv(f) for f in av_files], ignore_index=True)

av_names = sorted(av_df['Player'].dropna().unique().tolist())
print(f"Total unique players in AV CSVs: {len(av_names)}")

Total unique players in AV CSVs: 11895


In [17]:
api_names = set(season_df['player'].dropna().unique())
av_names_set = set(av_names)

matched       = av_names_set & api_names
only_in_av    = av_names_set - api_names
only_in_api   = api_names - av_names_set

print(f"Exact matches:        {len(matched)}")
print(f"Only in AV CSVs:      {len(only_in_av)}")
print(f"Only in API (2024):   {len(only_in_api)}")

matched_df     = season_df[season_df['player'].isin(matched)][['player','position','team']].drop_duplicates().sort_values('player')
only_in_av_df  = pd.DataFrame(sorted(only_in_av), columns=['player'])
only_in_api_df = pd.DataFrame(sorted(only_in_api), columns=['player'])

print("\n--- Sample: matched players ---")
display(matched_df.head(20))

print("\n--- Sample: in AV CSVs but NOT in 2024 API ---")
display(only_in_av_df.head(20))

print("\n--- Sample: in 2024 API but NOT in AV CSVs ---")
display(only_in_api_df.head(20))

Exact matches:        813
Only in AV CSVs:      11082
Only in API (2024):   12530

--- Sample: matched players ---


,player,position,team
6940,Aaron Banks,DB,Richmond
36757,Aaron Beasley,LB,Tennessee
75871,Aaron Smith,LB,South Carolina State
124321,Aaron Smith,S,New Mexico
75535,Abdul Carter,DE,Penn State
131456,Adam Jones,RB,Montana State
36389,Adam Jones,WR,Arkansas State
48286,Adin Huntington,DL,UL Monroe
12096,Adisa Isaac,DE,Penn State
47077,Adonai Mitchell,WR,Texas



--- Sample: in AV CSVs but NOT in 2024 API ---


,player
0,A'Shawn Robinson
1,A.J. Bouye
2,A.J. Brown
3,A.J. Cann
4,A.J. Davis
5,A.J. Derby
6,A.J. Edds
7,A.J. Epenesa
8,A.J. Feeley
9,A.J. Francis



--- Sample: in 2024 API but NOT in AV CSVs ---


,player
0,A'Khori Jones
1,A'Khoury Lyde
2,A'Marion McCoy
3,A'Marion Peterson
4,A'Mauri Washington
5,A.D. Diamond
6,A.J. Ackerman
7,A.J. Barner
8,A.J. Campbell
9,A.J. Coons


In [ ]:
import glob

av_files = glob.glob('../scraping_av/data/*.csv')
av_df = pd.concat([pd.read_csv(f) for f in av_files], ignore_index=True)

av_names = sorted(av_df['Player'].dropna().unique().tolist())
print(f"Total unique players in AV CSVs: {len(av_names)}")
av_names_set = set(av_names)

years = range(2013, 2000, -1)

for y in years:
    all_stats_raw = stats_api.get_player_season_stats(year=y)

    season_df = pd.DataFrame([{
        'player_id':  s.player_id,
        'player':     s.player,
        'position':   s.position,
        'team':       s.team,
        'conference': s.conference,
        'category':   s.category,
        'stat_type':  s.stat_type,
        'stat':       s.stat,
    } for s in all_stats_raw])

    print(f"Total rows: {len(season_df)}")
    print(f"Categories: {sorted(season_df['category'].unique())}")
    print(f"Teams: {season_df['team'].nunique()}")
    season_df.head(20)
    
    api_names = set(season_df['player'].dropna().unique())
    matched       = av_names_set & api_names
    
    matched_df     = season_df[season_df['player'].isin(matched)]
    matched_df.to_csv(f'raw_cfb/matched_players_{y}.csv', index=False)

## Data Cleaning

In [24]:
import re

# Rate stats recomputed from summed counts; never summed directly
RATE_MAP = {
    'passing_pct':       ('passing_completions', 'passing_att'),
    'passing_ypa':       ('passing_yds',          'passing_att'),
    'receiving_ypr':     ('receiving_yds',         'receiving_rec'),
    'rushing_ypc':       ('rushing_yds',           'rushing_car'),
    'kicking_pct':       ('kicking_fgm',           'kicking_fga'),
    'punting_ypp':       ('punting_yds',            'punting_no'),
    'kickreturns_avg':   ('kickreturns_yds',        'kickreturns_no'),
    'puntreturns_avg':   ('puntreturns_yds',        'puntreturns_no'),
    'interceptions_avg': ('interceptions_yds',      'interceptions_int'),
}

def build_player_df(csv_paths):
    """
    Takes a list of raw CFB CSV paths (long format: player_id, category, stat_type, stat).
    Returns one row per player_id with:
      - identity: player, position, latest_team, last_year, career_years
      - counting stats: summed across years
      - LONG (best single-play) stats: max across years
      - rate stats: recomputed from summed counts
    """
    frames = []
    for path in csv_paths:
        year = int(re.search(r'(\d{4})', path).group(1))
        df = pd.read_csv(path)
        df['year'] = year
        frames.append(df)
    raw = pd.concat(frames, ignore_index=True)

    # Prefixed column name: "passing_yds", "kickreturns_avg", etc.
    raw['col']  = (raw['category'].str.lower() + '_' +
                   raw['stat_type'].str.strip().str.lower().str.replace(' ', '_'))
    raw['stat'] = pd.to_numeric(raw['stat'], errors='coerce')

    all_stat_cols = raw['col'].unique().tolist()
    long_cols  = [c for c in all_stat_cols if c.endswith('_long')]
    rate_cols  = list(RATE_MAP.keys())
    sum_cols   = [c for c in all_stat_cols if c not in long_cols and c not in rate_cols]

    # Latest team/position/name; career span
    identity = (
        raw.sort_values('year')
        .groupby('player_id')
        .agg(
            player       = ('player',   'last'),
            position     = ('position', 'last'),
            latest_team  = ('team',     'last'),
            last_year    = ('year',     'max'),
            career_years = ('year',     'nunique'),
        )
        .reset_index()
    )

    # Pivot to (player_id, year) x stat_col, then aggregate across years
    wide = raw.pivot_table(
        index=['player_id', 'year'],
        columns='col',
        values='stat',
        aggfunc='first',
    ).reset_index()

    agg_dict = {
        **{c: 'sum' for c in sum_cols  if c in wide.columns},
        **{c: 'max' for c in long_cols if c in wide.columns},
    }
    stats_agg = wide.groupby('player_id').agg(agg_dict).reset_index()

    result = identity.merge(stats_agg, on='player_id', how='left')

    # Recompute rate stats from summed counts
    for rate_col, (num_col, den_col) in RATE_MAP.items():
        if num_col in result.columns and den_col in result.columns:
            result[rate_col] = result[num_col].div(result[den_col]).where(result[den_col] > 0)

    return result


# ── Validate on 2024 only ─────────────────────────────────────────────────────
df_2024 = build_player_df(['../data/raw_cfb/matched_players_2024.csv'])
print(f"Players : {len(df_2024)}")
print(f"Columns : {len(df_2024.columns)}")
print(sorted(df_2024.columns.tolist()))
df_2024.head(10)

Players : 585
Columns : 60
['career_years', 'defensive_pd', 'defensive_qb_hur', 'defensive_sacks', 'defensive_solo', 'defensive_td', 'defensive_tfl', 'defensive_tot', 'fumbles_fum', 'fumbles_lost', 'fumbles_rec', 'interceptions_avg', 'interceptions_int', 'interceptions_td', 'interceptions_yds', 'kicking_fga', 'kicking_fgm', 'kicking_long', 'kicking_pct', 'kicking_pts', 'kicking_xpa', 'kicking_xpm', 'kickreturns_avg', 'kickreturns_long', 'kickreturns_no', 'kickreturns_td', 'kickreturns_yds', 'last_year', 'latest_team', 'passing_att', 'passing_completions', 'passing_int', 'passing_pct', 'passing_td', 'passing_yds', 'passing_ypa', 'player', 'player_id', 'position', 'punting_in_20', 'punting_long', 'punting_no', 'punting_tb', 'punting_yds', 'punting_ypp', 'puntreturns_avg', 'puntreturns_long', 'puntreturns_no', 'puntreturns_td', 'puntreturns_yds', 'receiving_long', 'receiving_rec', 'receiving_td', 'receiving_yds', 'receiving_ypr', 'rushing_car', 'rushing_long', 'rushing_td', 'rushing_yds',

,player_id,player,position,latest_team,last_year,career_years,receiving_rec,receiving_td,receiving_yds,kickreturns_no,...,kicking_long,passing_pct,passing_ypa,receiving_ypr,rushing_ypc,kicking_pct,punting_ypp,kickreturns_avg,puntreturns_avg,interceptions_avg
0,548696,Malik Jackson,TE,Towson,2024,1,2.0,0.0,24.0,0.0,...,NaN,NaN,NaN,12.000000,NaN,NaN,NaN,NaN,NaN,NaN
1,560147,Kevin Johnson,WR,Hampton,2024,1,12.0,1.0,131.0,1.0,...,NaN,NaN,NaN,10.916667,NaN,NaN,NaN,17.0,NaN,NaN
2,4036672,Alex Smith,P,Georgia Southern,2024,1,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,42.741379,NaN,NaN,NaN
3,4360689,Tyler Shough,QB,Louisville,2024,1,0.0,0.0,0.0,0.0,...,NaN,0.627249,8.213368,NaN,0.452381,NaN,NaN,NaN,NaN,NaN
4,4361090,Ale Kaho,LB,UCLA,2024,1,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4383431,Cameron Lewis,WR,East Tennessee State,2024,1,19.0,2.0,207.0,0.0,...,NaN,NaN,NaN,10.894737,NaN,NaN,NaN,NaN,NaN,NaN
6,4426513,Trikweze Bridges,DB,Florida,2024,1,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
7,4426535,Theo Wease,WR,Missouri,2024,1,60.0,4.0,884.0,0.0,...,NaN,NaN,NaN,14.733333,NaN,NaN,NaN,NaN,NaN,NaN
8,4426542,Darius Alexander,DT,Toledo,2024,1,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,58.0
9,4426623,Brant Banks,OL,Rice,2024,1,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
# ── Pool all years (2004–2024) into a single player DataFrame ─────────────────
all_csv_paths = sorted(glob.glob('../data/raw_cfb/matched_players_*.csv'))
print(f"Loading {len(all_csv_paths)} files: {[p.split('/')[-1] for p in all_csv_paths]}")

df_all = build_player_df(all_csv_paths)

print(f"\nPlayers  : {len(df_all)}")
print(f"Columns  : {len(df_all.columns)}")
print(f"Year range: {int(df_all['last_year'].min())} – {int(df_all['last_year'].max())}")
print(f"Career years distribution:\n{df_all['career_years'].value_counts().sort_index()}")

df_all.head(10)

Loading 21 files: ['matched_players_2004.csv', 'matched_players_2005.csv', 'matched_players_2006.csv', 'matched_players_2007.csv', 'matched_players_2008.csv', 'matched_players_2009.csv', 'matched_players_2010.csv', 'matched_players_2011.csv', 'matched_players_2012.csv', 'matched_players_2013.csv', 'matched_players_2014.csv', 'matched_players_2015.csv', 'matched_players_2016.csv', 'matched_players_2017.csv', 'matched_players_2018.csv', 'matched_players_2019.csv', 'matched_players_2020.csv', 'matched_players_2021.csv', 'matched_players_2022.csv', 'matched_players_2023.csv', 'matched_players_2024.csv']

Players  : 6304
Columns  : 60
Year range: 2004 – 2024
Career years distribution:
1    1187
2    1134
3    1589
4    1831
5     504
6      59
Name: career_years, dtype: int64


,player_id,player,position,latest_team,last_year,career_years,interceptions_int,interceptions_td,interceptions_yds,passing_att,...,punting_long,passing_pct,passing_ypa,receiving_ypr,rushing_ypc,kicking_pct,punting_ypp,kickreturns_avg,puntreturns_avg,interceptions_avg
0,27200,Michael Johnson,WR,UNLV,2011,2,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,9.193548,3.909091,NaN,NaN,NaN,2.652174,NaN
1,29649,Jamaal Jackson,RB,Delaware State,2015,2,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,-5.000000,1.526316,NaN,NaN,15.0,NaN,NaN
2,106327,Jason Johnson,DB,Murray State,2018,2,1.0,0.0,0.0,0.0,...,NaN,NaN,NaN,2.000000,NaN,NaN,NaN,NaN,NaN,0.000000
3,118118,J.J. Jones,S,NC State,2004,1,1.0,0.0,2.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.000000
4,118214,Cody Davis,S,Texas Tech,2012,2,4.0,1.0,88.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.000000
5,123339,Ryan Donahue,P,Iowa,2010,4,0.0,0.0,0.0,0.0,...,82.0,NaN,NaN,NaN,-4.000000,NaN,41.888446,NaN,NaN,NaN
6,135811,Justin Miller,TE,Clemson,2004,1,3.0,0.0,14.0,1.0,...,NaN,1.0,2.0,NaN,3.000000,NaN,NaN,NaN,12.071429,4.666667
7,136743,Spencer Larsen,WR,Arizona,2007,2,2.0,0.0,3.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.500000
8,138728,Tyler Smith,RB,Rice,2011,4,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,7.118644,4.766667,NaN,NaN,NaN,NaN,NaN
9,146099,Brian Smith,WR,Minnesota,2016,1,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,16.411765,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
df_all.to_csv('clean_cfb/05_04_all_players_2004_2024.csv', index=False)

In [18]:
season_df[season_df['player'].isin(matched)].to_csv('raw_cfb/matched_players_2023.csv', index=False)

In [23]:
import glob

av_files = glob.glob('../scraping_av/data/*.csv')
av_df = pd.concat([pd.read_csv(f) for f in av_files], ignore_index=True)

av_names = sorted(av_df['Player'].dropna().unique().tolist())
print(f"Total unique players in AV CSVs: {len(av_names)}")
av_names_set = set(av_names)

years = range(2013, 2000, -1)

for y in years:
    all_stats_raw = stats_api.get_player_season_stats(year=y)

    season_df = pd.DataFrame([{
        'player_id':  s.player_id,
        'player':     s.player,
        'position':   s.position,
        'team':       s.team,
        'conference': s.conference,
        'category':   s.category,
        'stat_type':  s.stat_type,
        'stat':       s.stat,
    } for s in all_stats_raw])

    print(f"Total rows: {len(season_df)}")
    print(f"Categories: {sorted(season_df['category'].unique())}")
    print(f"Teams: {season_df['team'].nunique()}")
    season_df.head(20)
    
    api_names = set(season_df['player'].dropna().unique())
    matched       = av_names_set & api_names
    
    matched_df     = season_df[season_df['player'].isin(matched)]
    matched_df.to_csv(f'raw_cfb/matched_players_{y}.csv', index=False)

Total unique players in AV CSVs: 11895
Total rows: 39157
Categories: ['interceptions', 'kickReturns', 'kicking', 'passing', 'puntReturns', 'punting', 'receiving', 'rushing']
Teams: 209
Total rows: 34562
Categories: ['interceptions', 'kickReturns', 'kicking', 'passing', 'puntReturns', 'punting', 'receiving', 'rushing']
Teams: 202
Total rows: 32332
Categories: ['interceptions', 'kickReturns', 'kicking', 'passing', 'puntReturns', 'punting', 'receiving', 'rushing']
Teams: 191
Total rows: 31955
Categories: ['interceptions', 'kickReturns', 'kicking', 'passing', 'puntReturns', 'punting', 'receiving', 'rushing']
Teams: 168
Total rows: 30901
Categories: ['interceptions', 'kickReturns', 'kicking', 'passing', 'puntReturns', 'punting', 'receiving', 'rushing']
Teams: 133
Total rows: 18110
Categories: ['interceptions', 'kickReturns', 'kicking', 'passing', 'puntReturns', 'punting', 'receiving', 'rushing']
Teams: 122
Total rows: 10131
Categories: ['interceptions', 'kickReturns', 'kicking', 'passing', 

KeyError: 'category'